# Edgar Allan Poe–Style Text Generation

This notebook fine-tunes a small Transformer language model (GPT-2) on a corpus of Edgar Allan Poe’s public-domain writings (downloaded from Project Gutenberg) to generate new text in a Poe-like voice.

## Project recap
- **Goal:** learn a stylistic “Poe-ish” language model that can generate short passages on demand.
- **Pipeline:** download → clean & merge text → tokenize & chunk → fine-tune GPT-2 → generate samples.
- **Output:** a fine-tuned model checkpoint + example generations you can tweak with decoding parameters (temperature, top-p, max length).

## Why this project?
I love how Poe’s writing builds atmosphere—dread, beauty, and that uncanny rhythm that feels like a candle flickering in a storm. This project is a practical NLP exercise (data cleaning, tokenization, training, evaluation-by-sampling) and a fun way to explore how modern language models can imitate literary style.


## Step 1: Combine and Clean the Texts

so here, i am going to programmatically access Edgar Allan Poe’s works from Project Gutenberg using the gutenbergpy library.

In [1]:
!pip install gutenbergpy

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.4/199.4 kB 6.4 MB/s eta 0:00:00
  Created wheel for httpsproxy-urllib2: filename=httpsproxy_urllib2-1.0-py3-none-any.whl size=29251 sha256=a11b6312dd234a202bb9af03c9c6b3b93405c447c08f1da1b6c59b4bbe96e7cb
  Stored in directory: /root/.cache/pip/wheels/1b/fa/c3/4c14e72101070c40b56c2bfb4617e510e68f121e4f736a5d2a
Successfully built httpsproxy-urllib2


In [2]:
from gutenbergpy.textget import get_text_by_id
import re

# Step 1: Define the list of Project Gutenberg IDs for Edgar Allan Poe's works
poe_work_ids = [
    2147,  # The Works of Edgar Allan Poe, Volume 1
    2148,  # The Works of Edgar Allan Poe, Volume 2
    2149,  # The Works of Edgar Allan Poe, Volume 3
    2150,  # The Works of Edgar Allan Poe, Volume 4
    2151   # The Works of Edgar Allan Poe, Volume 5
]

# Step 2: Download and clean the texts
def download_and_clean_poe_works(ids):
    combined_text = ""
    for book_id in ids:
        raw_text = get_text_by_id(book_id).decode('utf-8', errors='ignore')

        # Remove headers/footers
        text = re.sub(r"(\*{3,}.*?START OF.*?TEXT.*?\*{3,})", "", raw_text, flags=re.DOTALL)
        text = re.sub(r"(\*{3,}.*?END OF.*?TEXT.*?\*{3,})", "", text, flags=re.DOTALL)

        # Normalize whitespace
        text = re.sub(r"\s+", " ", text).strip()

        combined_text += text + "\""
        
    return combined_text

# Step 3: Save the combined text to a file
poe_combined_text = download_and_clean_poe_works(poe_work_ids)
output_file = "poe_combined.txt"
with open(output_file, "w", encoding="utf-8") as file:
    file.write(poe_combined_text)

print(f"Downloaded and cleaned works saved to {output_file}")

Downloaded and cleaned works saved to poe_combined.txt


## Step 2: Preprocess the Data

In [3]:
import re

# Step 1: Load the downloaded text
input_file = "poe_combined.txt"  # Replace with the file you saved earlier
with open(input_file, "r", encoding="utf-8") as file:
    raw_text = file.read()

# Step 2: Clean the text
def clean_text(text):
    """
    Cleans the raw text by removing metadata, extra spaces, and non-ASCII characters.
    Args:
        text (str): Raw text input.
    Returns:
        str: Cleaned text output.
    """
    # Remove Project Gutenberg metadata (start and end markers)
    text = re.sub(r"(\*{3,}.*?START OF.*?TEXT.*?\*{3,})", "", text, flags=re.DOTALL)
    text = re.sub(r"(\*{3,}.*?END OF.*?TEXT.*?\*{3,})", "", text, flags=re.DOTALL)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Remove non-ASCII characters
    text = re.sub(r"[^\x00-\x7F]+", "", text)

    return text

cleaned_text = clean_text(raw_text)

# Step 3: Split into manageable chunks
def split_into_chunks(text, max_length=1000):
    """
    Splits the text into smaller chunks of a specified maximum length.
    Args:
        text (str): Cleaned text input.
        max_length (int): Maximum number of characters per chunk.
    Returns:
        list of str: List of text chunks.
    """
    sentences = re.split(r'(?<=[.!?]) +', text)  # Split into sentences
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_length:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk:  # Add the last chunk
        chunks.append(current_chunk.strip())

    return chunks

chunks = split_into_chunks(cleaned_text)

# Step 4: Save the preprocessed chunks
output_file = "poe_preprocessed.txt"
with open(output_file, "w", encoding="utf-8") as file:
    for chunk in chunks:
        file.write(chunk + "\\")

print(f"Preprocessed text saved to {output_file}")
print(f"Number of chunks: {len(chunks)}")

Preprocessed text saved to poe_preprocessed.txt
Number of chunks: 2882


## Step 3: Train a Text-Generating Model

things we'll need to do:

1. Set up the environment: Install necessary libraries.
2. Prepare the dataset: Load and tokenize the text.
3. Fine-tune GPT-2: Train the model on the Poe dataset.
4. Generate text: Test the model by generating Edgar Allan Poe-style text.

In [4]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments
from datasets import load_dataset
from transformers import TextDataset, DataCollatorForLanguageModeling

# Step 1: Load the dataset and tokenizer
file_path = '/kaggle/input/poe-texts/poe_combined.txt'  # Your dataset file path
block_size = 128  # Set block size according to your needs

# Load the tokenizer with a custom cache directory
tokenizer = GPT2Tokenizer.from_pretrained('gpt2', cache_dir='/kaggle/working/cache')

# Tokenize and load the dataset with a custom cache directory
def load_dataset(file_path, block_size):
    return TextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=block_size,
        cache_dir='/kaggle/working/cache'  # Cache directory set to writable location
    )

dataset = load_dataset(file_path, block_size)

# Step 2: Set up the data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # GPT2 does not use masked language modeling
)

# Step 3: Set up the model
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Step 4: Configure the training arguments
training_args = TrainingArguments(
    output_dir='./fine_tuned_model',  # Output directory
    overwrite_output_dir=True,
    num_train_epochs=3,  # Increased epochs for more training (previously 1)
    per_device_train_batch_size=4,  # Adjust batch size if needed
    save_steps=500,  # Save the model every 500 steps
    save_total_limit=2,  # Keep only the last 2 models
)

# Step 5: Set up the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

# Step 6: Train the model
trainer.train()

# Step 7: Save the fine-tuned model and tokenizer
model.save_pretrained('./fine_tuned_model')
tokenizer.save_pretrained('./fine_tuned_model')

print("Fine-tuning complete. Model and tokenizer saved!")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss
500,3.829300
1000,3.569600
1500,3.466000


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0

Fine-tuning complete. Model and tokenizer saved!


## Step 4: Generate Edgar Allan Poe-Style Text

and now, the moment of truth!

In [5]:
# After fine-tuning is complete
from transformers import pipeline

# Step 1: Load the fine-tuned model and tokenizer
model_path = './fine_tuned_model'  # Path to the saved fine-tuned model
model = GPT2LMHeadModel.from_pretrained(model_path)
tokenizer = GPT2Tokenizer.from_pretrained(model_path)

# Step 2: Set up the text generation pipeline
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

# Step 3: Generate text based on a prompt
prompt = "Once upon a midnight dreary, while I pondered, weak and weary,"  # Example prompt inspired by Poe's work
generated_text = generator(prompt, max_length=200, num_return_sequences=1, no_repeat_ngram_size=2)

# Step 4: Display the generated text (this will be the content you can screenshot)
print("Generated Text: ")
print(generated_text[0]['generated_text'])

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Generated Text: 
Once upon a midnight dreary, while I pondered, weak and weary, upon the dark shore of a beautiful lake in the southern part of the village, there I arose to my feet, and said unto them with profound speech: “You shall wake up in a dream and come into the world, as you were before. Thus, then, I say, may the dead come in, the deceased, by the light of thy life.” They all smiled, but for the moment, with an expressionless and wan countenance. Then they took the oath, shook hands with each other, turned the clock, shut up the door, went out to sleep, without speaking, gazed among the long, lofty shoreline, where some distant and deep sea lay, waiting for me at midnight. It was at dusk, after the departure of that hideous sea, that I heard the voice of an old woman, calling, ‘Let me hear her!’ ’


In [25]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, pipeline
from IPython.display import display, HTML

# Load the fine-tuned model and tokenizer
model = GPT2LMHeadModel.from_pretrained('./fine_tuned_model')
tokenizer = GPT2Tokenizer.from_pretrained('./fine_tuned_model')

# Set up the text generation pipeline
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

# Generate text
generated_text = generator("Once upon a midnight dreary, ", max_length=500, num_return_sequences=1)[0]['generated_text']

# Display the generated text in a styled box for screenshot purposes
html_output = f"""
<div style="border: 2px solid #ddd; border-radius: 10px; padding: 20px; background-color: #f4f4f4; 
            font-family: 'Courier New', monospace; font-size: 16px; white-space: pre-wrap; word-wrap: break-word;">
    <strong>Generated Text:</strong><br>
    <pre>{generated_text}</pre>
</div>
"""

# Display the styled box
display(HTML(html_output))


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
